In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

BRONZE_TABLE = "taxi_nyc.bronze.yellow_trips"
SILVER_TABLE = "taxi_nyc.silver.yellow_trips"
CHECKPOINT_PATH = "/Volumes/taxi_nyc/silver/pipeline_state/checkpoints/yellow_trips/"

rename_dict = {
    'VendorID': 'vendor_id',
    'tpep_pickup_datetime': 'pickup_at',
    'tpep_dropoff_datetime': 'dropoff_at',
    'RatecodeID': 'rate_code_id',
    'PULocationID': 'pickup_location_id',
    'DOLocationID': 'dropoff_location_id',
    'payment_type': 'payment_type_id',
    'extra': 'extra_amount',
    'mta_tax': 'mta_tax_amount',
    'improvement_surcharge': 'improvement_surcharge_amount',
    'congestion_surcharge': 'congestion_surcharge_amount',
    'Airport_fee': 'airport_fee_amount'
}


# Verificação de dados resgatados
resgate = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_rescued_data").isNotNull())
    .groupBy('_source_file')
    .count()
    .collect()
) 

if resgate:
    detalhes = ", ".join(f"{r['_source_file']}: {r['count']} linhas" for r in resgate)
    raise ValueError(f"Dados resgatados encontrados na bronze. {detalhes}")


# Leitura incremental da bronze e transformações
df_silver = (
    spark.readStream.table(BRONZE_TABLE)
    .withColumnsRenamed(rename_dict) # Renomeando nomes das colunas
    .drop('_rescued_data', '_modified_at') # Removendo colunas desnecessárias
    .withColumn('is_flex_fare', col('payment_type_id') == 0) # Nova coluna para tipo especifico de pagamento que gera colunas NULLS
    .filter(col('dropoff_at') > col('pickup_at')) #Filtrando para que o dropoff seja depois do pickup sempre
    .filter(
        F.date_trunc('month', 'pickup_at').cast('date')
        == F.to_date(F.regexp_extract("_source_file", r'(\d{4}-\d{2})', 1), 'yyyy-MM')
        ) # Filtrando para que o pickup seja de acordo com o periodo do arquivo
    )

# Escrita na silver
(
    df_silver.writeStream
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option('mergeSchema', 'true')
    .outputMode('append')
    .trigger(availableNow=True)
    .table(SILVER_TABLE)
    .awaitTermination()
)


            